# 01 — Stratified Sampling for Human Evaluation

This notebook selects **259 BDD scenarios** for human evaluation from the generation JSON files.

## Sampling design

- 5 models
- 3 techniques (`zero-shot`, `one-shot`, `few-shot`)
- 15 `model × technique` combinations
- 259 source cases
- 10 executions per combination

Each `case_id` appears **exactly once** in the human evaluation.

The distribution is balanced:

- each `model × technique` combination receives **17 or 18 cases**;
- each model receives **51 or 52 cases**;
- each technique receives **86 or 87 cases**;
- executions 1–10 are globally balanced at **25 or 26 occurrences**;
- the `seed` is fixed to allow exact reproduction of the sampling.

## Outputs

1. `chave_amostragem.json`  
   Researcher file. Stores `model`, `technique`, `execution`, `generation_id`, the original case, and the sampled BDD scenario.

2. `avaliacao_humana.json`  
   Blinded file for entering scores for:
   - Structure
   - Semantics
   - Details

The final score **is not calculated in this notebook**.

> **Compatibility note:** output JSON keys, metadata fields, categorical values, and output filenames remain exactly as in the original notebook so that previously generated files and downstream notebooks continue to work unchanged.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import json
import random

## 1. Configuration


In [ ]:
# Folder containing the 15 JSON files:
# 5 models × 3 techniques
JSON_FOLDER = Path("geracoes")

TOTAL_CASES = 259
NUMBER_OF_MODELS = 5
NUMBER_OF_TECHNIQUES = 3
NUMBER_OF_EXECUTIONS = 10

# Fixed seed for reproducibility
SEED = 20260826

SAMPLING_KEY_FILE = Path("chave_amostragem.json")
HUMAN_EVALUATION_FILE = Path("avaliacao_humana.json")

randomizer = random.Random(SEED)

## 2. Helper Functions


In [ ]:
def save_json(path, data):
    with open(path, "w", encoding="utf-8") as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2
        )


def normalize_text(text):
    if text is None:
        return None
    return str(text).strip()


def create_balanced_quotas(models, techniques, total, rng):
    """
    Creates quotas for model × technique combinations.

    For 259 cases and 15 combinations:
    - 11 combinations receive 17
    - 4 combinations receive 18

    The four extra cases are distributed so that:
    - four different models receive +1;
    - among the techniques, one receives two extras and the others receive one extra.
    """
    cells = [
        (model, technique)
        for model in models
        for technique in techniques
    ]

    base_quota = total // len(cells)
    remainder = total % len(cells)

    quotas = {
        cell: base_quota
        for cell in cells
    }

    if remainder == 0:
        return quotas

    # For this design: 4 extras, 5 models, and 3 techniques.
    if remainder <= len(models):
        extra_models = rng.sample(models, remainder)

        extra_techniques = []

        # Most balanced possible distribution across techniques.
        while len(extra_techniques) < remainder:
            block = list(techniques)
            rng.shuffle(block)
            extra_techniques.extend(block)

        extra_techniques = extra_techniques[:remainder]
        rng.shuffle(extra_techniques)

        for model, technique in zip(extra_models, extra_techniques):
            quotas[(model, technique)] += 1
    else:
        # Generic fallback.
        extra_cells = rng.sample(cells, remainder)
        for cell in extra_cells:
            quotas[cell] += 1

    return quotas


def create_execution_plan(quotas, number_of_executions, rng, max_attempts=5000):
    """
    Creates a globally balanced execution plan.

    Each model × technique combination initially receives one occurrence
    of each execution 1..10. Additional occurrences are distributed
    so that, across the 259 cases, each execution appears 25 or 26 times.

    Returns:
    - global_target: desired count for each execution;
    - plan: list of executions to be used in each stratum.
    """
    cells = list(quotas.keys())
    number_of_cells = len(cells)
    total = sum(quotas.values())

    global_base = total // number_of_executions
    global_remainder = total % number_of_executions

    global_target = {
        execution: global_base
        for execution in range(1, number_of_executions + 1)
    }

    executions_with_extra = rng.sample(
        list(global_target.keys()),
        global_remainder
    )

    for execution in executions_with_extra:
        global_target[execution] += 1

    # Each execution already appears once in each of the 15 cells.
    repetition_targets = {
        execution: global_target[execution] - number_of_cells
        for execution in global_target
    }

    repetitions_per_cell = {
        cell: quotas[cell] - number_of_executions
        for cell in cells
    }

    if sum(repetition_targets.values()) != sum(repetitions_per_cell.values()):
        raise RuntimeError("Inconsistency while building the execution plan.")

    for _ in range(max_attempts):
        remaining = dict(repetition_targets)
        chosen_repetitions = {}

        cell_order = sorted(
            cells,
            key=lambda c: (-repetitions_per_cell[c], rng.random())
        )

        failed = False

        for cell in cell_order:
            number_of_repetitions = repetitions_per_cell[cell]

            candidates = [
                execution
                for execution, remaining_count in remaining.items()
                if remaining_count > 0
            ]

            if len(candidates) < number_of_repetitions:
                failed = True
                break

            # Prioritize executions that still need to appear more often.
            candidates.sort(
                key=lambda e: (-remaining[e], rng.random())
            )

            chosen = candidates[:number_of_repetitions]
            chosen_repetitions[cell] = chosen

            for execution in chosen:
                remaining[execution] -= 1

        if failed:
            continue

        if not all(value == 0 for value in remaining.values()):
            continue

        plan = {}

        for cell in cells:
            executions = list(range(1, number_of_executions + 1))
            executions.extend(chosen_repetitions[cell])
            rng.shuffle(executions)
            plan[cell] = executions

        return global_target, plan

    raise RuntimeError(
        "Could not build a balanced execution plan."
    )

## 3. Load Generation JSON Files


In [ ]:
json_files = sorted(JSON_FOLDER.glob("*.json"))

if not json_files:
    raise RuntimeError(
        f"No JSON file found in: {JSON_FOLDER.resolve()}"
    )

data_by_stratum = {}

for path in json_files:
    with open(path, "r", encoding="utf-8") as file:
        data = json.load(file)

    model = data["model"]
    technique = data["technique"]
    stratum_key = (model, technique)

    if stratum_key in data_by_stratum:
        raise ValueError(
            f"More than one file found for {model} / {technique}"
        )

    cases = {}

    for case in data["cases"]:
        case_id = case["case_id"]

        generations = {}

        for generation in case["generations"]:
            execution = int(generation["execution"])

            if execution in generations:
                raise ValueError(
                    f"{path.name}: duplicate execution {execution} in {case_id}"
                )

            generations[execution] = {
                "generation_id": generation["generation_id"],
                "gherkin": generation["gherkin"]
            }

        found_executions = set(generations.keys())
        expected_executions = set(
            range(1, NUMBER_OF_EXECUTIONS + 1)
        )

        if found_executions != expected_executions:
            raise ValueError(
                f"{path.name}: {case_id} does not contain exactly "
                f"executions 1..{NUMBER_OF_EXECUTIONS}. "
                f"Found: {sorted(found_executions)}"
            )

        cases[case_id] = {
            "source_id": case.get("source_id"),
            "source_line": case.get("source_line"),
            "original_case": case["original_case"],
            "generations": generations
        }

    data_by_stratum[stratum_key] = {
        "arquivo": path.name,
        "cases": cases
    }

print(f"Files loaded: {len(json_files)}")

## 4. Validate Models, Techniques, and Source Cases


In [ ]:
models = sorted({
    model
    for model, technique in data_by_stratum
})

techniques = sorted({
    technique
    for model, technique in data_by_stratum
})

print("Models:")
for model in models:
    print(" -", model)

print("\nTechniques:")
for technique in techniques:
    print(" -", technique)

if len(models) != NUMBER_OF_MODELS:
    raise ValueError(
        f"Expected {NUMBER_OF_MODELS} models, "
        f"but {len(models)} were found."
    )

if len(techniques) != NUMBER_OF_TECHNIQUES:
    raise ValueError(
        f"Expected {NUMBER_OF_TECHNIQUES} techniques, "
        f"but {len(techniques)} were found."
    )

expected_strata_count = (
    NUMBER_OF_MODELS * NUMBER_OF_TECHNIQUES
)

if len(data_by_stratum) != expected_strata_count:
    raise ValueError(
        f"Expected {expected_strata_count} combinations "
        f"model × technique, but "
        f"{len(data_by_stratum)} were found."
    )

first_stratum = next(iter(data_by_stratum))
reference_cases = set(
    data_by_stratum[first_stratum]["cases"].keys()
)

if len(reference_cases) != TOTAL_CASES:
    raise ValueError(
        f"Expected {TOTAL_CASES} source cases, "
        f"but {len(reference_cases)} were found."
    )

# Also validates whether source_id and original_case are consistent
# across all models/techniques.
reference_details = {
    case_id: (
        data_by_stratum[first_stratum]["cases"][case_id]["source_id"],
        normalize_text(
            data_by_stratum[first_stratum]["cases"][case_id]["original_case"]
        )
    )
    for case_id in reference_cases
}

for stratum_key, content in data_by_stratum.items():
    current_cases = set(content["cases"].keys())

    if current_cases != reference_cases:
        missing = sorted(reference_cases - current_cases)
        extra = sorted(current_cases - reference_cases)

        raise ValueError(
            f"The cases do not match in {stratum_key}.\n"
            f"Missing: {missing}\n"
            f"Extra: {extra}"
        )

    for case_id in reference_cases:
        current = content["cases"][case_id]

        current_detail = (
            current["source_id"],
            normalize_text(current["original_case"])
        )

        if current_detail != reference_details[case_id]:
            raise ValueError(
                f"Source-case inconsistency in {case_id}, "
                f"stratum {stratum_key}."
            )

print(f"\nValidation completed: {TOTAL_CASES} identical cases across the 15 strata.")

## 5. Create Balanced Quotas by Model × Technique


In [ ]:
quotas = create_balanced_quotas(
    models=models,
    techniques=techniques,
    total=TOTAL_CASES,
    rng=randomizer
)

print("Model × technique quotas:\n")

for model in models:
    print(model)
    for technique in techniques:
        print(f"  {technique}: {quotas[(model, technique)]}")

print("\nTotal:", sum(quotas.values()))

## 6. Create a Globally Balanced Execution Plan


In [ ]:
execution_target, execution_plan = create_execution_plan(
    quotas=quotas,
    number_of_executions=NUMBER_OF_EXECUTIONS,
    rng=randomizer
)

print("Target execution distribution:")
for execution in range(1, NUMBER_OF_EXECUTIONS + 1):
    print(f"Execution {execution}: {execution_target[execution]}")

## 7. Sample the Model × Technique Combination for Each Source Case


In [ ]:
slots = []

for stratum, quantity in quotas.items():
    slots.extend([stratum] * quantity)

if len(slots) != TOTAL_CASES:
    raise RuntimeError("Incorrect number of slots.")

randomizer.shuffle(slots)

case_ids = sorted(reference_cases)
randomizer.shuffle(case_ids)

assignments = defaultdict(list)

for case_id, stratum in zip(case_ids, slots):
    assignments[stratum].append(case_id)

if sum(len(ids) for ids in assignments.values()) != TOTAL_CASES:
    raise RuntimeError("Error assigning cases to strata.")

## 8. Select the Exact Execution for Each Case


In [ ]:
selected = []

for stratum, case_ids_in_stratum in assignments.items():
    model, technique = stratum

    case_ids_in_stratum = list(case_ids_in_stratum)
    randomizer.shuffle(case_ids_in_stratum)

    executions = list(execution_plan[stratum])

    if len(case_ids_in_stratum) != len(executions):
        raise RuntimeError(
            f"The number of cases and executions does not match in {stratum}."
        )

    for case_id, execution in zip(case_ids_in_stratum, executions):
        case = data_by_stratum[stratum]["cases"][case_id]
        generation = case["generations"][execution]

        selected.append({
            "case_id": case_id,
            "source_id": case["source_id"],
            "source_line": case["source_line"],
            "original_case": case["original_case"],
            "model": model,
            "technique": technique,
            "execution": execution,
            "generation_id": generation["generation_id"],
            "gherkin": generation["gherkin"]
        })

if len(selected) != TOTAL_CASES:
    raise RuntimeError(
        f"{len(selected)} cases were selected, "
        f"but {TOTAL_CASES} were expected."
    )

# Each source case must appear exactly once.
selected_ids = [item["case_id"] for item in selected]

if len(set(selected_ids)) != TOTAL_CASES:
    raise RuntimeError("There are duplicate case_id values in the sample.")

# Each generation_id must be unique.
generation_ids = [item["generation_id"] for item in selected]

if len(set(generation_ids)) != TOTAL_CASES:
    raise RuntimeError("There are duplicate generation_id values in the sample.")

print("Selection completed successfully.")

## 9. Validate Final Balancing


In [ ]:
stratum_counts = Counter(
    (item["model"], item["technique"])
    for item in selected
)

model_counts = Counter(
    item["model"]
    for item in selected
)

technique_counts = Counter(
    item["technique"]
    for item in selected
)

execution_counts = Counter(
    item["execution"]
    for item in selected
)

for stratum, expected in quotas.items():
    found = stratum_counts[stratum]

    if found != expected:
        raise RuntimeError(
            f"Incorrect quota in {stratum}: "
            f"expected {expected}, found {found}."
        )

for execution, expected in execution_target.items():
    found = execution_counts[execution]

    if found != expected:
        raise RuntimeError(
            f"Execution {execution}: expected {expected}, "
            f"found {found}."
        )

print("By model:")
for model in models:
    print(f"  {model}: {model_counts[model]}")

print("\nBy technique:")
for technique in techniques:
    print(f"  {technique}: {technique_counts[technique]}")

print("\nBy execution:")
for execution in range(1, NUMBER_OF_EXECUTIONS + 1):
    print(f"  Execution {execution}: {execution_counts[execution]}")

## 10. Shuffle Evaluation Order and Create Blind IDs


In [ ]:
randomizer.shuffle(selected)

for index, item in enumerate(selected, start=1):
    item["avaliacao_id"] = f"AV_{index:03d}"

print("Example IDs:")
for item in selected[:5]:
    print(item["avaliacao_id"], "->", item["case_id"])

## 11. Generate `chave_amostragem.json`


In [ ]:
sampling_key = {
    "metadata": {
        "seed": SEED,
        "total_casos": TOTAL_CASES,
        "quantidade_modelos": NUMBER_OF_MODELS,
        "quantidade_tecnicas": NUMBER_OF_TECHNIQUES,
        "quantidade_execucoes": NUMBER_OF_EXECUTIONS,
        "metodo": "amostragem balanceada por modelo x tecnica",
        "unidade_amostral": "uma geracao BDD por caso-fonte",
        "chave_para_metricas": "generation_id",
        "observacao_metricas": (
            "generation_id identifica exatamente caso, modelo, tecnica e execucao"
        )
    },
    "selecoes": []
}

for item in selected:
    sampling_key["selecoes"].append({
        "avaliacao_id": item["avaliacao_id"],
        "case_id": item["case_id"],
        "source_id": item["source_id"],
        "source_line": item["source_line"],
        "original_case": item["original_case"],
        "model": item["model"],
        "technique": item["technique"],
        "execution": item["execution"],
        "generation_id": item["generation_id"],
        "gherkin": item["gherkin"]
    })

save_json(
    SAMPLING_KEY_FILE,
    sampling_key
)

print(f"Generated: {SAMPLING_KEY_FILE.resolve()}")

## 12. Generate `avaliacao_humana.json`


In [ ]:
human_evaluation = {
    "metadata": {
        "total_casos": TOTAL_CASES,
        "escala": {
            "minimo": 0,
            "maximo": 10
        },
        "criterios": {
            "estrutura": {
                "peso": 4,
                "descricao": (
                    "Avaliar clareza e adequacao estrutural do cenario BDD/Gherkin."
                )
            },
            "semantica": {
                "peso": 4,
                "descricao": (
                    "Avaliar se o BDD preserva corretamente a intencao "
                    "do caso de teste original."
                )
            },
            "detalhes": {
                "peso": 2,
                "descricao": (
                    "Avaliar a presenca de dados, condicoes e detalhes "
                    "relevantes para o comportamento testado."
                )
            }
        },
        "observacao": (
            "Preencher apenas estrutura, semantica e detalhes. "
            "A nota final sera calculada no Notebook 02."
        )
    },
    "avaliacoes": []
}

for item in selected:
    human_evaluation["avaliacoes"].append({
        "avaliacao_id": item["avaliacao_id"],
        "case_id": item["case_id"],
        "caso_original": item["original_case"],
        "bdd_gerado": item["gherkin"],
        "avaliacao": {
            "estrutura": None,
            "semantica": None,
            "detalhes": None
        }
    })

save_json(
    HUMAN_EVALUATION_FILE,
    human_evaluation
)

print(f"Generated: {HUMAN_EVALUATION_FILE.resolve()}")

## 13. Final Summary


In [ ]:
print("=" * 70)
print("SAMPLING COMPLETED")
print("=" * 70)

print(f"Seed: {SEED}")
print(f"Total selected: {len(selected)}")
print(f"Unique Case IDs: {len(set(selected_ids))}")
print(f"Unique Generation IDs: {len(set(generation_ids))}")

print("\nModel × technique:")
for model in models:
    print(f"\n{model}")
    for technique in techniques:
        print(
            f"  {technique}: "
            f"{stratum_counts[(model, technique)]}"
        )

print("\nExecutions:")
for execution in range(1, NUMBER_OF_EXECUTIONS + 1):
    print(
        f"  Execution {execution}: "
        f"{execution_counts[execution]}"
    )

print("\nFiles:")
print(f" - {SAMPLING_KEY_FILE}")
print(f" - {HUMAN_EVALUATION_FILE}")